In [1]:
import warnings
warnings.filterwarnings("ignore")
import random
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import StandardScaler


In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [6]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [7]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [8]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [9]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]

In [10]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [11]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [12]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [13]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 374)
y_train:  (139,)


(99, 376)

# Feature Selection: RENT

In [14]:
selected_features = ["shape_Sphericity",
                     "glrlm_HighGrayLevelRunEmphasis_PET_c04",
                     "shape_MajorAxisLength",
                     "LBP_102_PET"]

# Selecting features in the DataFrame
X_rent = X[selected_features]

In [15]:
# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Standardization

In [16]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = StandardScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [17]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [18]:
X_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.761164,16.969770,42.073251,0.000000
1,0.697049,15.598394,24.613845,0.000000
2,0.565792,17.334294,48.030294,0.000034
3,0.684364,14.009277,25.589900,0.000000
4,0.503142,21.202180,34.684750,0.000199
...,...,...,...,...
134,0.742102,16.110383,33.069705,0.000000
135,0.722918,21.249575,41.043692,0.000000
136,0.652963,14.873884,36.618802,0.000000
137,0.724255,21.648860,45.870392,0.000000


In [19]:
X_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,1.075042,0.081641,-0.098422,-0.562251
1,0.242767,-0.395928,-1.209119,-0.562251
2,-1.461057,0.208583,0.280542,-0.202838
3,0.078112,-0.949323,-1.147026,-0.562251
4,-2.274305,1.555538,-0.568449,1.514080
...,...,...,...,...
134,0.827589,-0.217633,-0.671191,-0.562251
135,0.578577,1.572043,-0.163918,-0.562251
136,-0.329498,-0.648232,-0.445412,-0.562251
137,0.595927,1.711090,0.143137,-0.562251


In [20]:
MAASTRO_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.668072,19.034156,50.002093,0.000026
1,0.669961,11.392444,41.753334,0.000167
2,0.624081,14.567421,44.375483,0.000057
3,0.577624,13.477331,46.115989,0.000000
4,0.630933,16.365554,54.394967,0.000000
...,...,...,...,...
94,0.671754,17.492295,34.218615,0.000000
95,0.632189,14.267900,51.046869,0.000000
96,0.645548,12.143752,50.417953,0.000000
97,0.727488,15.300229,44.901412,0.000000


In [21]:
MAASTRO_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,-0.133381,0.800544,0.405980,-0.288893
1,-0.108854,-1.860610,-0.118774,1.182201
2,-0.704414,-0.754955,0.048037,0.033974
3,-1.307469,-1.134568,0.158761,-0.562251
4,-0.615466,-0.128772,0.685437,-0.562251
...,...,...,...,...
94,-0.085581,0.263605,-0.598102,-0.562251
95,-0.599161,-0.859260,0.472444,-0.562251
96,-0.425762,-1.598975,0.432435,-0.562251
97,0.637891,-0.499761,0.081495,-0.562251


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [22]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 01:05:04,495] A new study created in memory with name: no-name-becc15a4-3dfa-474d-8393-62be883e5fd0


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6796536796536796
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.7341772151898734


[I 2024-04-16 01:05:16,357] A new study created in memory with name: no-name-7dd2fa87-8c6e-40e9-99db-004a3906f0b5


Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 01:05:16,319] Trial 0 finished with value: 0.7358264523738474 and parameters: {}. Best is trial 0 with value: 0.7358264523738474.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7358264523738474], datetime_start=datetime.datetime(2024, 4, 16, 1, 5, 4, 657951), datetime_complete=datetime.datetime(2024, 4, 16, 1, 5, 16, 317569), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7358264523738474


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.17932951816141288
Fold 2 IBS: 0.15173912855888336
Fold 3 IBS: 0.17662964771229905
Fold 4 IBS: 0.15303709600503956
Fold 5 IBS: 0.19681553106978894
[I 2024-04-16 01:05:17,203] Trial 0 finished with value: 0.17151018430148474 and parameters: {}. Best is trial 0 with value: 0.17151018430148474.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.17151018430148474], datetime_start=datetime.datetime(2024, 4, 16, 1, 5, 16, 433269), datetime_complete=datetime.datetime(2024, 4, 16, 1, 5, 17, 203079), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.17151018430148474


In [23]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [24]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.736
train_ibs:  0.172


#### Test

In [25]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [26]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.567
IBS score: 0.259


In [27]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [28]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [29]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:05:17,907] A new study created in memory with name: no-name-dea07611-7263-4e80-afa3-5efc23c5815d


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6493506493506493
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7818627450980392
Fold 4 C-index: 0.7151898734177216


[I 2024-04-16 01:05:18,415] A new study created in memory with name: no-name-3baece87-cb3c-4618-b2c4-044e4a07479c


Fold 5 C-index: 0.7230046948356808
[I 2024-04-16 01:05:18,392] Trial 0 finished with value: 0.7238815925404183 and parameters: {}. Best is trial 0 with value: 0.7238815925404183.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7238815925404183], datetime_start=datetime.datetime(2024, 4, 16, 1, 5, 17, 976929), datetime_complete=datetime.datetime(2024, 4, 16, 1, 5, 18, 392048), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7238815925404183


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2139765155205475
Fold 2 IBS: 0.22157790435913943
Fold 3 IBS: 0.20453593833829126
Fold 4 IBS: 0.22473803160496592
Fold 5 IBS: 0.21812430497307525
[I 2024-04-16 01:05:19,028] Trial 0 finished with value: 0.21659053895920385 and parameters: {}. Best is trial 0 with value: 0.21659053895920385.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659053895920385], datetime_start=datetime.datetime(2024, 4, 16, 1, 5, 18, 548812), datetime_complete=datetime.datetime(2024, 4, 16, 1, 5, 19, 28582), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659053895920385


In [30]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [31]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.724
train_ibs:  0.217


#### Test

In [32]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [33]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.578


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [34]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [35]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:05:19,422] A new study created in memory with name: no-name-4f335f19-d004-48b8-8e52-b4ce5f60a12e


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.729957805907173


[I 2024-04-16 01:05:20,425] A new study created in memory with name: no-name-3fec6f31-e61e-40e0-ab66-e06146d6a6a3


Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 01:05:20,361] Trial 0 finished with value: 0.7357337800920462 and parameters: {}. Best is trial 0 with value: 0.7357337800920462.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7357337800920462], datetime_start=datetime.datetime(2024, 4, 16, 1, 5, 19, 488725), datetime_complete=datetime.datetime(2024, 4, 16, 1, 5, 20, 361192), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7357337800920462


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.17976231773999546
Fold 2 IBS: 0.15088281580357382
Fold 3 IBS: 0.17576178549951946
Fold 4 IBS: 0.15354534218421234
Fold 5 IBS: 0.19571810296360162
[I 2024-04-16 01:05:21,276] Trial 0 finished with value: 0.17113407283818055 and parameters: {}. Best is trial 0 with value: 0.17113407283818055.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.17113407283818055], datetime_start=datetime.datetime(2024, 4, 16, 1, 5, 20, 483010), datetime_complete=datetime.datetime(2024, 4, 16, 1, 5, 21, 275759), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.17113407283818055


In [36]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [37]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.736
train_ibs:  0.171


#### Test 

In [38]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [39]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.566


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.257


In [40]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [41]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:05:22,054] A new study created in memory with name: no-name-011c9ef3-0a4a-4d9a-9a22-b77194339687


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 01:05:23,381] Trial 0 finished with value: 0.7365995809578472 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7365995809578472.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 01:05:24,509] Trial 1 finished with value: 0.7367412285259652 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.7367412285259652.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 01:05:25,739] Trial 2 finished with value: 0.7358483713831081 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 1 with value

Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 01:05:46,506] Trial 24 finished with value: 0.7365995809578472 and parameters: {'l1_ratio': 0.8080056249917272}. Best is trial 3 with value: 0.7375799731147099.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 01:05:47,276] Trial 25 finished with value: 0.7365995809578472 and parameters: {'l1_ratio': 0.628676464772803}. Best is trial 3 with value: 0.7375799731147099.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 01:05:48,418] Trial 26 finished with value: 0.7367141722489089 and parameters: {'l1_ratio': 0.3527001258697712}. Best is trial 3 with value: 0.7375799731147099.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892

Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 01:06:13,763] Trial 49 finished with value: 0.7367141722489089 and parameters: {'l1_ratio': 0.355745098499332}. Best is trial 3 with value: 0.7375799731147099.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 01:06:14,502] Trial 50 finished with value: 0.7375799731147099 and parameters: {'l1_ratio': 0.5938078108685914}. Best is trial 3 with value: 0.7375799731147099.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.6948356807511737
[I 2024-04-16 01:06:15,203] Trial 51 finished with value: 0.7375799731147099 and parameters: {'l1_ratio': 0.5434891196509528}. Best is trial 3 with value: 0.7375799731147099.
Fold 1 C-index: 0.6926

...

Fold 1 IBS: 0.17961702079353403
Fold 2 IBS: 0.15095065897682072
Fold 3 IBS: 0.1758817854688703
Fold 4 IBS: 0.15358600376544532
Fold 5 IBS: 0.19566662202018217
[I 2024-04-16 01:07:24,502] Trial 0 finished with value: 0.1711404182049705 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.1711404182049705.
Fold 1 IBS: 0.17921854874687435
Fold 2 IBS: 0.1510190050679179
Fold 3 IBS: 0.17631580674739586
Fold 4 IBS: 0.1536432060050057
Fold 5 IBS: 0.1957526240358902
[I 2024-04-16 01:07:26,311] Trial 1 finished with value: 0.1711898381206168 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.1711404182049705.
Fold 1 IBS: 0.17913179009993588
Fold 2 IBS: 0.1510564341999031
Fold 3 IBS: 0.17641151273022218
Fold 4 IBS: 0.15370123481959697
Fold 5 IBS: 0.19578302112960017
[I 2024-04-16 01:07:27,452] Trial 2 finished with value: 0.17121679859585165 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.1711404182049705.


Fold 1 IBS: 0.17964727559710011
Fold 2 IBS: 0.15090070946174614
Fold 3 IBS: 0.17586860056125767
Fold 4 IBS: 0.15358513339564106
Fold 5 IBS: 0.1957101009421332
[I 2024-04-16 01:07:49,693] Trial 25 finished with value: 0.17114236399157562 and parameters: {'l1_ratio': 0.7647388825054626}. Best is trial 16 with value: 0.17112592887357167.
Fold 1 IBS: 0.17959195732865083
Fold 2 IBS: 0.1509332457520489
Fold 3 IBS: 0.17595664209883857
Fold 4 IBS: 0.1536038983124461
Fold 5 IBS: 0.19573415329574262
[I 2024-04-16 01:07:51,296] Trial 26 finished with value: 0.1711639793575454 and parameters: {'l1_ratio': 0.6477441511102914}. Best is trial 16 with value: 0.17112592887357167.
Fold 1 IBS: 0.17970857045989355
Fold 2 IBS: 0.15091923918873767
Fold 3 IBS: 0.1757847418170072
Fold 4 IBS: 0.15356547507642446
Fold 5 IBS: 0.19564228408637446
[I 2024-04-16 01:07:52,720] Trial 27 finished with value: 0.17112406212568748 and parameters: {'l1_ratio': 0.8313376819651622}. Best is trial 27 with value: 0.1711240621

Fold 1 IBS: 0.17970700862703434
Fold 2 IBS: 0.15091817989655337
Fold 3 IBS: 0.17578541708073786
Fold 4 IBS: 0.1535669941206311
Fold 5 IBS: 0.19564007146711287
[I 2024-04-16 01:08:23,244] Trial 50 finished with value: 0.17112353423841392 and parameters: {'l1_ratio': 0.827239601583696}. Best is trial 35 with value: 0.1711234970681431.
Fold 1 IBS: 0.17970553221714716
Fold 2 IBS: 0.15091717852898873
Fold 3 IBS: 0.17578605580278628
Fold 4 IBS: 0.15356843080687638
Fold 5 IBS: 0.19574176735273358
[I 2024-04-16 01:08:24,653] Trial 51 finished with value: 0.17114379294170642 and parameters: {'l1_ratio': 0.8234003712590802}. Best is trial 35 with value: 0.1711234970681431.
Fold 1 IBS: 0.17964176753949349
Fold 2 IBS: 0.15089662605437165
Fold 3 IBS: 0.17587099137263867
Fold 4 IBS: 0.15356177486974165
Fold 5 IBS: 0.19570222254923034
[I 2024-04-16 01:08:25,645] Trial 52 finished with value: 0.17113467647709518 and parameters: {'l1_ratio': 0.7513882197173816}. Best is trial 35 with value: 0.171123497

Fold 1 IBS: 0.179629267151078
Fold 2 IBS: 0.150959089149268
Fold 3 IBS: 0.17587643706925069
Fold 4 IBS: 0.15357400365471247
Fold 5 IBS: 0.1956841777174661
[I 2024-04-16 01:08:50,744] Trial 75 finished with value: 0.17114459494835504 and parameters: {'l1_ratio': 0.7226548259502308}. Best is trial 71 with value: 0.17112325096289532.
Fold 1 IBS: 0.17959734661613208
Fold 2 IBS: 0.15093707446029403
Fold 3 IBS: 0.1759544122469034
Fold 4 IBS: 0.15359845573489758
Fold 5 IBS: 0.19574151389648767
[I 2024-04-16 01:08:52,156] Trial 76 finished with value: 0.17116576059094296 and parameters: {'l1_ratio': 0.6579092078440976}. Best is trial 71 with value: 0.17112325096289532.
Fold 1 IBS: 0.1797286640165961
Fold 2 IBS: 0.1509328619835906
Fold 3 IBS: 0.17577609206475558
Fold 4 IBS: 0.15354600188520437
Fold 5 IBS: 0.1956706843309654
[I 2024-04-16 01:08:53,954] Trial 77 finished with value: 0.17113086085622245 and parameters: {'l1_ratio': 0.8876747639518479}. Best is trial 71 with value: 0.17112325096289

Fold 5 IBS: 0.19571187732417217
[I 2024-04-16 01:09:26,039] Trial 99 finished with value: 0.1711427806157631 and parameters: {'l1_ratio': 0.7678130961399826}. Best is trial 71 with value: 0.17112325096289532.


* Best trial for IBS: 
 FrozenTrial(number=71, state=TrialState.COMPLETE, values=[0.17112325096289532], datetime_start=datetime.datetime(2024, 4, 16, 1, 8, 44, 496390), datetime_complete=datetime.datetime(2024, 4, 16, 1, 8, 45, 575037), params={'l1_ratio': 0.8250557880797453}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=71, value=None)


* Best Score for IBS: 
 0.17112325096289532


In [42]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [43]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.738
train_ibs:  0.171


#### Test

In [44]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [45]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.5513596376059829)

test_cindex : 0.566


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.8250557880797453)

test_ibs:  0.257


In [46]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [47]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 01:09:27,246] A new study created in memory with name: no-name-5c721061-1922-4a49-a853-bae192e24925


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7467532467532467
Fold 2 C-index: 0.8125
Fold 3 C-index: 0.7769607843137255
Fold 4 C-index: 0.6814345991561181
Fold 5 C-index: 0.7699530516431925
[I 2024-04-16 01:09:39,113] Trial 0 finished with value: 0.7575203363732566 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7575203363732566.
Fold 1 C-index: 0.7445887445887446
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.75
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7699530516431925
[I 2024-04-16 01:09:48,459] Trial 1 finished with value: 0.750478431579118 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_features': 'sqrt', 'mi

Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.7991071428571429
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.7341772151898734
Fold 5 C-index: 0.7746478873239436
[I 2024-04-16 01:11:43,852] Trial 16 finished with value: 0.765813594477808 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 16, 'max_depth': 20, 'n_estimators': 5, 'oob_score': True, 'max_samples': 0.8484814588885312, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.08082385841318356, 'warm_start': True}. Best is trial 14 with value: 0.7907144637948861.
Fold 1 C-index: 0.7597402597402597
Fold 2 C-index: 0.8526785714285714
Fold 3 C-index: 0.7867647058823529
Fold 4 C-index: 0.7236286919831224
Fold 5 C-index: 0.8028169014084507
[I 2024-04-16 01:11:46,099] Trial 17 finished with value: 0.7851258260885514 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 15, 'min_samples_leaf': 15, 'max_depth': 17, 'n_estimators': 106, 'oob_score': True, 'max_samples': 0.98170723046990

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.7658227848101266
Fold 5 C-index: 0.8356807511737089
[I 2024-04-16 01:12:11,078] Trial 31 finished with value: 0.7982641653072841 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 14, 'min_samples_leaf': 7, 'max_depth': 8, 'n_estimators': 75, 'oob_score': True, 'max_samples': 0.745146001885883, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.05174429879930956, 'warm_start': True}. Best is trial 31 with value: 0.7982641653072841.
Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.7742616033755274
Fold 5 C-index: 0.8309859154929577
[I 2024-04-16 01:12:12,155] Trial 32 finished with value: 0.8032544311992128 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 14, 'min_samples_leaf': 7, 'max_depth': 9, 'n_estimators': 78, 'oob_score': True, 'max_samples': 0.7397628776532771, 'm

...

Fold 5 IBS: 0.17698987901902175
[I 2024-04-16 01:26:13,863] Trial 31 finished with value: 0.1756223060197624 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 1, 'n_estimators': 495, 'oob_score': False, 'max_samples': 0.6141886010044044, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.050889504115644586}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17921499719523448
Fold 2 IBS: 0.14457659998302025
Fold 3 IBS: 0.18892068480933402
Fold 4 IBS: 0.1881785321534129
Fold 5 IBS: 0.17366220512930025
[I 2024-04-16 01:26:31,937] Trial 32 finished with value: 0.17491060385406038 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 473, 'oob_score': False, 'max_samples': 0.7145937356077748, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.040680386261001684}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17839650482493216
Fold 2 IBS: 0.

Fold 1 IBS: 0.16980857091041326
Fold 2 IBS: 0.15719115142034723
Fold 3 IBS: 0.1767449129839921
Fold 4 IBS: 0.19510331227382113
Fold 5 IBS: 0.1831755952242518
[I 2024-04-16 01:29:36,741] Trial 47 finished with value: 0.17640470856256513 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 20, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 479, 'oob_score': False, 'max_samples': 0.8914383434724882, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.24824101632322468}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.19349637716238946
Fold 2 IBS: 0.18021514079410297
Fold 3 IBS: 0.1797766992061227
Fold 4 IBS: 0.20805913242700583
Fold 5 IBS: 0.19958771263889286
[I 2024-04-16 01:29:50,436] Trial 48 finished with value: 0.19222701244570276 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 6, 'max_depth': 3, 'n_estimators': 439, 'oob_score': False, 'max_samples': 0.9967788477130068, 'max_features': 'auto', 'min_weight_fraction_

Fold 5 IBS: 0.17601995444169838
[I 2024-04-16 01:33:34,719] Trial 62 finished with value: 0.1766496500814655 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 9, 'max_depth': 3, 'n_estimators': 298, 'oob_score': True, 'max_samples': 0.6969767031486694, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.11947149985769817}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17325965795999854
Fold 2 IBS: 0.16035062956465257
Fold 3 IBS: 0.17539882197990753
Fold 4 IBS: 0.193362443315604
Fold 5 IBS: 0.1796554283287958
[I 2024-04-16 01:33:53,750] Trial 63 finished with value: 0.17640539622979168 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 15, 'min_samples_leaf': 10, 'max_depth': 1, 'n_estimators': 311, 'oob_score': True, 'max_samples': 0.6304198495967799, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.16702154024396243}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17417602273955624
Fold 2 IBS: 0.1480464

Fold 1 IBS: 0.16818314645426713
Fold 2 IBS: 0.15369061634554518
Fold 3 IBS: 0.18213459050433092
Fold 4 IBS: 0.19123211966917278
Fold 5 IBS: 0.17979973767912885
[I 2024-04-16 01:39:16,570] Trial 78 finished with value: 0.17500804213048896 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 487, 'oob_score': False, 'max_samples': 0.4897248144615137, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.053996872945124036}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.1682237549262069
Fold 2 IBS: 0.15295334400832536
Fold 3 IBS: 0.18214608764478374
Fold 4 IBS: 0.19238806563941324
Fold 5 IBS: 0.18028732465861996
[I 2024-04-16 01:39:35,857] Trial 79 finished with value: 0.17519971537546983 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 11, 'max_depth': 11, 'n_estimators': 500, 'oob_score': False, 'max_samples': 0.49575398669591153, 'max_features': 'auto', 'min_weight_frac

Fold 5 IBS: 0.1747309238915691
[I 2024-04-16 01:45:38,246] Trial 93 finished with value: 0.1752048898255005 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 10, 'max_depth': 4, 'n_estimators': 437, 'oob_score': False, 'max_samples': 0.5630698614554802, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.0944985972942633}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17026943747099219
Fold 2 IBS: 0.15255352423829832
Fold 3 IBS: 0.18160069813968083
Fold 4 IBS: 0.19096533404342483
Fold 5 IBS: 0.1822030877543707
[I 2024-04-16 01:45:57,198] Trial 94 finished with value: 0.17551841632935336 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 410, 'oob_score': False, 'max_samples': 0.5751544947591161, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.14409687866880386}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17475332961241374
Fold 2 IBS: 0.1452

In [48]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [49]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.846
train_ibs:  0.171


#### Test

In [50]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [51]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=13, max_features=None, max_leaf_nodes=12,
                     max_samples=0.48042970450395145, min_samples_leaf=1,
                     min_samples_split=4,
                     min_weight_fraction_leaf=0.0025914663927405594,
                     n_estimators=289, random_state=123, warm_start=True)

test_cindex:  0.602


RandomSurvivalForest(max_depth=1, max_features='auto', max_leaf_nodes=18,
                     max_samples=0.7351810575897255, min_samples_leaf=11,
                     min_samples_split=2,
                     min_weight_fraction_leaf=0.008231293935378081,
                     n_estimators=2, random_state=123)

test_ibs:  0.233


In [52]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [53]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:47:35,924] A new study created in memory with name: no-name-258e7d2c-e9ca-42eb-bfd0-111fdaeb516c


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.70995670995671
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.7257383966244726
Fold 5 C-index: 0.755868544600939
[I 2024-04-16 01:47:41,000] Trial 0 finished with value: 0.763120153205612 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.763120153205612.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 01:47:51,494] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. Be

Fold 4 C-index: 0.7046413502109705
Fold 5 C-index: 0.7136150234741784
[I 2024-04-16 01:49:19,309] Trial 15 finished with value: 0.7545769177214963 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.7656344262692516.
Fold 1 C-index: 0.6645021645021645
Fold 2 C-index: 0.8191964285714286
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.7088607594936709
Fold 5 C-index: 0.676056338028169
[I 2024-04-16 01:49:23,552] Trial 16 finished with value: 0.7335270596877141 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is

Fold 1 C-index: 0.7186147186147186
Fold 2 C-index: 0.84375
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.7257383966244726
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 01:50:20,182] Trial 30 finished with value: 0.7619426335421767 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 12, 'min_samples_leaf': 1, 'max_depth': 9, 'n_estimators': 473, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.8382504072008882, 'min_weight_fraction_leaf': 0.09166005683798992}. Best is trial 19 with value: 0.7663281634231028.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.84375
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7046413502109705
Fold 5 C-index: 0.7276995305164319
[I 2024-04-16 01:50:23,385] Trial 31 finished with value: 0.7601328692954678 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 16, 'min_samples_leaf': 15, 'max_depth': 3, 'n_estimators': 375, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_sa

...

Fold 1 IBS: 0.18923073618097216
Fold 2 IBS: 0.17632006586851645
Fold 3 IBS: 0.18707067058836963
Fold 4 IBS: 0.19307240934736852
Fold 5 IBS: 0.19343354994654252
[I 2024-04-16 01:58:12,345] Trial 30 finished with value: 0.18782548638635385 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.18218668005091396.
Fold 1 IBS: 0.1949036294792145
Fold 2 IBS: 0.1886371865011974
Fold 3 IBS: 0.19117863815442443
Fold 4 IBS: 0.2001198684681372
Fold 5 IBS: 0.19929856633981852
[I 2024-04-16 01:58:21,204] Trial 31 finished with value: 0.19482757778855841 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 2, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 396, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.

Fold 1 IBS: 0.18107213977138975
Fold 2 IBS: 0.17375539456626857
Fold 3 IBS: 0.18442566650093445
Fold 4 IBS: 0.19068424307356363
Fold 5 IBS: 0.18831741970696433
[I 2024-04-16 02:00:46,326] Trial 45 finished with value: 0.18365097272382416 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 11, 'min_samples_leaf': 2, 'max_depth': 17, 'n_estimators': 366, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.9992649320581628, 'min_weight_fraction_leaf': 0.15624439720920977}. Best is trial 44 with value: 0.17800229474539986.
Fold 1 IBS: 0.19953982633918327
Fold 2 IBS: 0.19967504623742535
Fold 3 IBS: 0.19245042867286613
Fold 4 IBS: 0.2085347403492104
Fold 5 IBS: 0.20542311562702958
[I 2024-04-16 02:00:56,933] Trial 46 finished with value: 0.20112463144514292 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 10, 'min_samples_leaf': 4, 'max_depth': 15, 'n_estimators': 471, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.8

Fold 1 IBS: 0.17659457791938485
Fold 2 IBS: 0.15948702371575116
Fold 3 IBS: 0.18725320428102982
Fold 4 IBS: 0.18157027728642194
Fold 5 IBS: 0.18138733472020935
[I 2024-04-16 02:02:55,822] Trial 60 finished with value: 0.17725848358455942 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 15, 'n_estimators': 346, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7897702249078459, 'min_weight_fraction_leaf': 0.017004106952235656}. Best is trial 57 with value: 0.1764459724345519.
Fold 1 IBS: 0.17815851130122365
Fold 2 IBS: 0.15937759601308119
Fold 3 IBS: 0.18671233531081954
Fold 4 IBS: 0.18030514263303932
Fold 5 IBS: 0.18041925981213724
[I 2024-04-16 02:03:03,842] Trial 61 finished with value: 0.1769945690140602 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 15, 'n_estimators': 340, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.861

Fold 1 IBS: 0.21400073415496554
Fold 2 IBS: 0.2208296768873117
Fold 3 IBS: 0.20504439468604338
Fold 4 IBS: 0.22482948519570425
Fold 5 IBS: 0.21793226667188703
[I 2024-04-16 02:04:49,118] Trial 75 finished with value: 0.21652731151918242 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 6, 'min_samples_leaf': 7, 'max_depth': 11, 'n_estimators': 301, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.28939609623192764, 'min_weight_fraction_leaf': 0.38792353869222}. Best is trial 73 with value: 0.17587600438953827.
Fold 1 IBS: 0.18092605908791307
Fold 2 IBS: 0.1626714178491272
Fold 3 IBS: 0.1886398695722679
Fold 4 IBS: 0.18093264680250482
Fold 5 IBS: 0.18045471816506836
[I 2024-04-16 02:04:54,660] Trial 76 finished with value: 0.17872494229537628 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 4, 'min_samples_leaf': 5, 'max_depth': 14, 'n_estimators': 229, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.82756

Fold 1 IBS: 0.18417873567489768
Fold 2 IBS: 0.17215507074054423
Fold 3 IBS: 0.18759974758071274
Fold 4 IBS: 0.1889870504957803
Fold 5 IBS: 0.18950903071510605
[I 2024-04-16 02:06:34,802] Trial 90 finished with value: 0.1844859270414082 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 4, 'min_samples_leaf': 4, 'max_depth': 10, 'n_estimators': 262, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9722689668575789, 'min_weight_fraction_leaf': 0.013698489831834436}. Best is trial 73 with value: 0.17587600438953827.
Fold 1 IBS: 0.17828049657703648
Fold 2 IBS: 0.15670442146325114
Fold 3 IBS: 0.1885595963781235
Fold 4 IBS: 0.17837564239035944
Fold 5 IBS: 0.17924664597888262
[I 2024-04-16 02:06:43,449] Trial 91 finished with value: 0.17623336055753064 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 5, 'max_depth': 15, 'n_estimators': 308, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.83

In [54]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [55]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.791
train_ibs:  0.176


#### Test

In [56]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [57]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=19, max_features=1, max_leaf_nodes=12,
                   max_samples=0.21404335777324493, min_samples_leaf=1,
                   min_samples_split=4,
                   min_weight_fraction_leaf=0.0013133682808468272,
                   n_estimators=405, random_state=123, warm_start=True)

C-index score: 0.572


ExtraSurvivalTrees(max_depth=14, max_features=None, max_leaf_nodes=6,
                   max_samples=0.8396800309305413, min_samples_leaf=5,
                   min_samples_split=5,
                   min_weight_fraction_leaf=0.02494499808293557,
                   n_estimators=305, oob_score=True, random_state=123)

IBS: 0.221


In [58]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [59]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 02:07:45,817] A new study created in memory with name: no-name-0586cbbe-ae78-4adc-9047-d88d54968e14


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:08:22,371] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:08:44,456] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:19:02,960] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7610259439316011.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:20:18,579] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:33:26,637] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8840318412875596, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.27382248438555523, 'n_estimators': 385, 'criterion': 'squared_error', 'ccp_alpha': 2.0183033060186855, 'min_weight_fraction_leaf': 0.33643534713187806, 'max_features': 'auto', 'min_impurity_decrease': 5.889654690360788e-06, 'validation_fraction': 0.8062622793646869, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7610259439316011.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:34:36,890] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7565765917190008, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.4300954216773497, 'n_estimators': 440, 'criterion': 'friedman

...

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609557
[I 2024-04-16 04:22:01,750] Trial 44 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9992870370700113, 'learning_rate': 0.022847552015173876, 'dropout_rate': 0.1556807870961761, 'n_estimators': 451, 'criterion': 'squared_error', 'ccp_alpha': 1.6646055539220843, 'min_weight_fraction_leaf': 0.1403134453903068, 'max_features': 'auto', 'min_impurity_decrease': 2.7552659293421345e-07, 'validation_fraction': 0.8820166309308186, 'min_samples_split': 17, 'max_leaf_nodes': 14, 'min_samples_leaf': 10, 'max_depth': 2}. Best is trial 42 with value: 0.21286195228463267.
Fold 1 IBS: 0.21224782790603694
Fold 2 IBS: 0.2189260608342495
Fold 3 IBS: 0.20307124042767
Fold 4 IBS: 0.22313607826909793
Fold 5 IBS: 0.216439921823279
[I 2024-04-16 04:22:30,477] Trial 45 finished with value: 0.2147642258520667 and parameters: {'subsample': 0.8888212863898438, 'learning_rate': 0.0152498321107806

Fold 3 IBS: 0.20286393718880646
Fold 4 IBS: 0.22391249179729633
Fold 5 IBS: 0.21519882725417372
[I 2024-04-16 04:29:10,109] Trial 55 finished with value: 0.21426660724128474 and parameters: {'subsample': 0.9703353679292269, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.2676210325613897, 'n_estimators': 481, 'criterion': 'squared_error', 'ccp_alpha': 0.036860238643527846, 'min_weight_fraction_leaf': 0.21798842867076448, 'max_features': None, 'min_impurity_decrease': 4.086647023052284e-07, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 10, 'max_depth': 4}. Best is trial 42 with value: 0.21286195228463267.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-16 04:29:59,367] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.9556274374724505, 'learning_rate': 0.022777236

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-16 04:32:44,218] Trial 66 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9019393211486977, 'learning_rate': 0.09984676062907398, 'dropout_rate': 0.14697195606329974, 'n_estimators': 8, 'criterion': 'squared_error', 'ccp_alpha': 0.7042671412306138, 'min_weight_fraction_leaf': 0.07018529613097008, 'max_features': None, 'min_impurity_decrease': 3.137142110783047e-07, 'validation_fraction': 0.779814650235569, 'min_samples_split': 18, 'max_leaf_nodes': 13, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 57 with value: 0.21250989041794505.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-16 04:32:49,606] Trial 67 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.922035492452401, 'learning_rate': 0.021388128948998

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609557
[I 2024-04-16 04:34:16,652] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.834194505922225, 'learning_rate': 0.023289065494024846, 'dropout_rate': 0.5946683576609367, 'n_estimators': 105, 'criterion': 'friedman_mse', 'ccp_alpha': 5.8752783688356525, 'min_weight_fraction_leaf': 0.10946812472698092, 'max_features': 'log2', 'min_impurity_decrease': 3.863869195457471e-05, 'validation_fraction': 0.8140600247331974, 'min_samples_split': 16, 'max_leaf_nodes': 17, 'min_samples_leaf': 12, 'max_depth': 1}. Best is trial 70 with value: 0.21173544073366624.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-16 04:34:19,286] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9202091045129451, 'learning_rate': 0.019797687069

Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609582
[I 2024-04-16 04:35:09,117] Trial 88 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.45521815335975935, 'learning_rate': 0.024635453467641497, 'dropout_rate': 0.22592407076200724, 'n_estimators': 225, 'criterion': 'friedman_mse', 'ccp_alpha': 0.5932990711529844, 'min_weight_fraction_leaf': 0.05795880477433249, 'max_features': 0.1, 'min_impurity_decrease': 0.00010910005120952281, 'validation_fraction': 0.86583149343058, 'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 6, 'max_depth': 6}. Best is trial 82 with value: 0.2114835599960237.
Fold 1 IBS: 0.20406423720489414
Fold 2 IBS: 0.20607296701907554
Fold 3 IBS: 0.19811121797640907
Fold 4 IBS: 0.22011942798029224
Fold 5 IBS: 0.20776003972546755
[I 2024-04-16 04:35:13,130] Trial 89 finished with value: 0.2072255779812277 and parameters: {'subsample': 0.9081334064688743, 'learning_rate': 0.032246585462716706, 'dropout_rate': 0.10084377

In [60]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [61]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.765
train_ibs:  0.207


#### Test

In [62]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [63]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.046986154574499193,
                                 dropout_rate=0.9141084507237711,
                                 learning_rate=0.005228819896792163,
                                 max_leaf_nodes=13,
                                 min_impurity_decrease=0.00017501601948817654,
                                 min_samples_leaf=15, min_samples_split=11,
                                 min_weight_fraction_leaf=0.2724947676380759,
                                 n_estimators=395, random_state=123,
                                 subsample=0.6128161811459047,
                                 validation_fraction=0.542037097833947)

C-index score: 0.579


GradientBoostingSurvivalAnalysis(ccp_alpha=0.019065157478907357,
                                 dropout_rate=0.10084377376256726,
                                 learning_rate=0.032246585462716706,
                                 max_leaf_nodes=15,
                                 min_impurity_decrease=6.552978044824526e-05,
                                 min_samples_leaf=9, min_samples_split=18,
                                 min_weight_fraction_leaf=0.21843803937299006,
                                 n_estimators=92, random_state=123,
                                 subsample=0.9081334064688743,
                                 validation_fraction=0.8292287698601738)

IBS: 0.218


In [64]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [65]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 04:35:59,424] A new study created in memory with name: no-name-0394b27d-e6da-4c13-a930-ace2a25bf38e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 04:36:00,332] Trial 0 finished with value: 0.6925028294272052 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6925028294272052.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.75
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 04:36:07,667] Trial 1 finished with value: 0.6915224372703425 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6925028294272052.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.73708920

Fold 1 C-index: 0.5670995670995671
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.679324894514768
Fold 5 C-index: 0.7464788732394366
[I 2024-04-16 04:37:12,867] Trial 19 finished with value: 0.6987557369987656 and parameters: {'subsample': 0.2202898915217226, 'dropout_rate': 0.3156718779160812, 'n_estimators': 431, 'learning_rate': 0.01563167261765183}. Best is trial 14 with value: 0.7111502809928003.
Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.6751054852320675
Fold 5 C-index: 0.7511737089201878
[I 2024-04-16 04:37:15,639] Trial 20 finished with value: 0.6997977919753454 and parameters: {'subsample': 0.38161734041406226, 'dropout_rate': 0.22438470567364258, 'n_estimators': 245, 'learning_rate': 0.03827516477850919}. Best is trial 14 with value: 0.7111502809928003.
Fold 1 C-index: 0.5974025974025974
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7745098039215687


Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7464788732394366
[I 2024-04-16 04:38:38,864] Trial 38 finished with value: 0.6961416936829352 and parameters: {'subsample': 0.27261582706417964, 'dropout_rate': 0.8190477307045616, 'n_estimators': 231, 'learning_rate': 0.04270368088653838}. Best is trial 22 with value: 0.7112039412516981.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 04:38:42,048] Trial 39 finished with value: 0.6951295602764356 and parameters: {'subsample': 0.42175313697788064, 'dropout_rate': 0.3997102511114324, 'n_estimators': 277, 'learning_rate': 0.02944859066794525}. Best is trial 22 with value: 0.7112039412516981.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.75
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.73

...

Fold 2 IBS: 0.17621580201176304
Fold 3 IBS: 0.17642037928014906
Fold 4 IBS: 0.1933021033103859
Fold 5 IBS: 0.18417216136826517
[I 2024-04-16 04:44:43,927] Trial 19 finished with value: 0.18620993107053097 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.18620993107053097.
Fold 1 IBS: 0.20387936509804852
Fold 2 IBS: 0.1793124075420424
Fold 3 IBS: 0.17651390954439816
Fold 4 IBS: 0.19656951529579103
Fold 5 IBS: 0.18647847319032207
[I 2024-04-16 04:44:44,306] Trial 20 finished with value: 0.18855073413412043 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.18620993107053097.
Fold 1 IBS: 0.2014156239858501
Fold 2 IBS: 0.186025328567609
Fold 3 IBS: 0.18124723411340657
Fold 4 IBS: 0.20164577976082107
Fold 5 IBS: 0.1923852965944346
[I 2024-04-

Fold 2 IBS: 0.16157113568264034
Fold 3 IBS: 0.18180869186592977
Fold 4 IBS: 0.2047333534968014
Fold 5 IBS: 0.1741931240386079
[I 2024-04-16 04:45:01,839] Trial 38 finished with value: 0.18807952320663193 and parameters: {'subsample': 0.10045702238474481, 'dropout_rate': 0.13272164755980653, 'n_estimators': 82, 'learning_rate': 0.0684395247111694}. Best is trial 19 with value: 0.18620993107053097.
Fold 1 IBS: 0.22633764950020804
Fold 2 IBS: 0.1753146373948459
Fold 3 IBS: 0.17881883861510856
Fold 4 IBS: 0.20168491431079613
Fold 5 IBS: 0.17197177530632424
[I 2024-04-16 04:45:02,699] Trial 39 finished with value: 0.19082556302545656 and parameters: {'subsample': 0.167666800643059, 'dropout_rate': 0.11471626893495929, 'n_estimators': 100, 'learning_rate': 0.050430345636818495}. Best is trial 19 with value: 0.18620993107053097.
Fold 1 IBS: 0.2422412002312995
Fold 2 IBS: 0.2157930776654539
Fold 3 IBS: 0.18031024557675085
Fold 4 IBS: 0.2534886449121738
Fold 5 IBS: 0.18564670364656027
[I 2024-0

Fold 5 IBS: 0.17811616534317862
[I 2024-04-16 04:45:20,303] Trial 57 finished with value: 0.19620422631092646 and parameters: {'subsample': 0.2718885877959942, 'dropout_rate': 0.3400190341028076, 'n_estimators': 46, 'learning_rate': 0.09297626863772185}. Best is trial 41 with value: 0.18279696898170414.
Fold 1 IBS: 0.23662970506901534
Fold 2 IBS: 0.19727329320405082
Fold 3 IBS: 0.1930525473184726
Fold 4 IBS: 0.22924643889195298
Fold 5 IBS: 0.18343607043154547
[I 2024-04-16 04:45:21,684] Trial 58 finished with value: 0.20792761098300744 and parameters: {'subsample': 0.15211700057388067, 'dropout_rate': 0.29749461681784134, 'n_estimators': 128, 'learning_rate': 0.052849112789723526}. Best is trial 41 with value: 0.18279696898170414.
Fold 1 IBS: 0.20644167878414996
Fold 2 IBS: 0.2091292617508421
Fold 3 IBS: 0.19355020066897802
Fold 4 IBS: 0.21277104569319658
Fold 5 IBS: 0.2078375808249311
[I 2024-04-16 04:45:21,982] Trial 59 finished with value: 0.20594595354441955 and parameters: {'subsa

Fold 1 IBS: 0.20857973644761613
Fold 2 IBS: 0.1699443446840626
Fold 3 IBS: 0.17229814826540152
Fold 4 IBS: 0.18923467234725175
Fold 5 IBS: 0.17149001199096955
[I 2024-04-16 04:45:46,581] Trial 77 finished with value: 0.18230938274706032 and parameters: {'subsample': 0.12220726626359973, 'dropout_rate': 0.3022992602914743, 'n_estimators': 67, 'learning_rate': 0.052951683263892116}. Best is trial 77 with value: 0.18230938274706032.
Fold 1 IBS: 0.24216819173841428
Fold 2 IBS: 0.20216193415298328
Fold 3 IBS: 0.19215980224541818
Fold 4 IBS: 0.23811111014860895
Fold 5 IBS: 0.18050576760623235
[I 2024-04-16 04:45:47,712] Trial 78 finished with value: 0.21102136117833142 and parameters: {'subsample': 0.20767510402915473, 'dropout_rate': 0.3088829531372439, 'n_estimators': 129, 'learning_rate': 0.05306134761655199}. Best is trial 77 with value: 0.18230938274706032.
Fold 1 IBS: 0.22260632680353415
Fold 2 IBS: 0.18964762214328523
Fold 3 IBS: 0.17055630609791816
Fold 4 IBS: 0.22240457037821182
Fol

Fold 1 IBS: 0.23957162447856908
Fold 2 IBS: 0.2116279809548012
Fold 3 IBS: 0.1829969605048687
Fold 4 IBS: 0.22906250124122637
Fold 5 IBS: 0.18381146611337082
[I 2024-04-16 04:46:02,257] Trial 96 finished with value: 0.20941410665856722 and parameters: {'subsample': 0.2906503742188524, 'dropout_rate': 0.21627122392159215, 'n_estimators': 109, 'learning_rate': 0.054487916689644846}. Best is trial 83 with value: 0.18193609918243667.
Fold 1 IBS: 0.20174967150385922
Fold 2 IBS: 0.18152252410983202
Fold 3 IBS: 0.17926438853781912
Fold 4 IBS: 0.19807000889983026
Fold 5 IBS: 0.1862867633778203
[I 2024-04-16 04:46:02,827] Trial 97 finished with value: 0.18937867128583222 and parameters: {'subsample': 0.15644204623115698, 'dropout_rate': 0.17846494250737677, 'n_estimators': 75, 'learning_rate': 0.022115499272652852}. Best is trial 83 with value: 0.18193609918243667.
Fold 1 IBS: 0.20621289427842387
Fold 2 IBS: 0.17587073239319495
Fold 3 IBS: 0.17447698010049484
Fold 4 IBS: 0.19856583741456169
Fol

In [66]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [67]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.712
train_ibs:  0.182


#### Test

In [68]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [69]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.8703793504419942,
                                              learning_rate=0.09213034239034594,
                                              n_estimators=419,
                                              random_state=123,
                                              subsample=0.10218130137857191)

C-index score: 0.575


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.26481642929201277,
                                              learning_rate=0.047702833771498435,
                                              n_estimators=57, random_state=123,
                                              subsample=0.12485705742267929)

IBS: 0.222


In [70]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [71]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.846,1.0
ExtraSurvivalTrees,0.791,2.0
GradientBoosting,0.765,3.0
CoxElastic,0.738,4.0
CoxPH,0.736,5.5
CoxLasso,0.736,5.5
CoxRidge,0.724,7.0
ComponentwiseGradientBoosting,0.712,8.0


In [72]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
CoxLasso,0.171,2.0
CoxElastic,0.171,2.0
Randomsurvivalforest,0.171,2.0
CoxPH,0.172,4.0
ExtraSurvivalTrees,0.176,5.0
ComponentwiseGradientBoosting,0.182,6.0
GradientBoosting,0.207,7.0
CoxRidge,0.217,8.0


In [73]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.602,1.0
GradientBoosting,0.579,2.0
CoxRidge,0.578,3.0
ComponentwiseGradientBoosting,0.575,4.0
ExtraSurvivalTrees,0.572,5.0
CoxPH,0.567,6.0
CoxLasso,0.566,7.5
CoxElastic,0.566,7.5


In [74]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
GradientBoosting,0.218,1.0
CoxRidge,0.221,2.5
ExtraSurvivalTrees,0.221,2.5
ComponentwiseGradientBoosting,0.222,4.0
Randomsurvivalforest,0.233,5.0
CoxLasso,0.257,6.5
CoxElastic,0.257,6.5
CoxPH,0.259,8.0


In [77]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/os/standard/rent/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_os_standard_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [78]:
from datetime import date

current_date = date.today()
print(current_date)

2024-04-16
